# Train model từ Kaggle baseline + `src/forecasting.py`

Notebook này kết hợp 2 hướng:
1. **Kaggle baseline**: seasonal profile + tăng trưởng năm (trend)
2. **Pipeline mới** trong `src/forecasting.py`: Time-Series CV + baseline methods

Mục tiêu:
- Train và backtest cả 2 hướng
- So sánh MAE trên giai đoạn holdout gần nhất
- Xuất file dự báo theo phương án tốt hơn

In [ ]:
from pathlib import Path
import json


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'

sales_path = (PROCESSED_DIR / 'sales.csv') if (PROCESSED_DIR / 'sales.csv').exists() else (RAW_DIR / 'sales.csv')
template_path = (PROCESSED_DIR / 'sample_submission.csv') if (PROCESSED_DIR / 'sample_submission.csv').exists() else (RAW_DIR / 'sample_submission.csv')

print('sales_path   :', sales_path)
print('template_path:', template_path)
print('output_dir   :', PROCESSED_DIR)

In [ ]:
sales = pd.read_csv(sales_path, low_memory=False)
sales['Date'] = pd.to_datetime(sales['Date'], errors='coerce')
sales['Revenue'] = pd.to_numeric(sales['Revenue'], errors='coerce')
sales = sales.dropna(subset=['Date', 'Revenue']).sort_values('Date').drop_duplicates('Date', keep='last').reset_index(drop=True)

template = pd.read_csv(template_path, low_memory=False)
template['Date'] = pd.to_datetime(template['Date'], errors='coerce')
if template['Date'].isna().any():
    raise ValueError('Template có Date không hợp lệ.')

print('sales shape   :', sales.shape)
print('template shape:', template.shape)
display(sales.head(3))

## A) Train theo Kaggle baseline (seasonal profile + yearly trend)

In [ ]:
kb = sales.copy()
kb['year'] = kb['Date'].dt.year
kb['month'] = kb['Date'].dt.month
kb['day'] = kb['Date'].dt.day

if 'COGS' in kb.columns:
    kb['COGS'] = pd.to_numeric(kb['COGS'], errors='coerce')
else:
    ratio = 0.1
    kb['COGS'] = kb['Revenue'] * ratio

annual = kb.groupby('year')[['Revenue', 'COGS']].sum()
annual

In [ ]:
# Geometric mean growth (dùng toàn bộ năm có sẵn, bỏ năm đầu tiên khi tính pct_change)
yoy_rev = annual['Revenue'].pct_change().dropna()
yoy_cogs = annual['COGS'].pct_change().dropna()
growth_rev = float((1 + yoy_rev).prod() ** (1 / len(yoy_rev))) if len(yoy_rev) else 1.0
growth_cogs = float((1 + yoy_cogs).prod() ** (1 / len(yoy_cogs))) if len(yoy_cogs) else 1.0

annual_means = kb.groupby('year')[['Revenue', 'COGS']].transform('mean')
kb['rev_norm'] = kb['Revenue'] / annual_means['Revenue']
kb['cogs_norm'] = kb['COGS'] / annual_means['COGS']

seasonal = (
    kb.groupby(['month', 'day'])[['rev_norm', 'cogs_norm']]
    .mean()
    .reset_index()
)

print('growth_rev :', growth_rev)
print('growth_cogs:', growth_cogs)
display(seasonal.head())

In [ ]:
# Backtest nhanh: holdout 120 ngày cuối
holdout_size = 120
train_kb = kb.iloc[:-holdout_size].copy()
valid_kb = kb.iloc[-holdout_size:].copy()

base_year = int(train_kb['year'].max())
annual_train = train_kb.groupby('year')[['Revenue', 'COGS']].sum()
base_rev = float(annual_train.loc[base_year, 'Revenue'] / 365)
base_cogs = float(annual_train.loc[base_year, 'COGS'] / 365)

valid_kb = valid_kb.merge(
    seasonal.rename(columns={'rev_norm': 'seasonal_rev_norm', 'cogs_norm': 'seasonal_cogs_norm'}),
    on=['month', 'day'],
    how='left',
)
valid_kb['seasonal_rev_norm'] = valid_kb['seasonal_rev_norm'].fillna(1.0)
valid_kb['years_ahead'] = valid_kb['year'] - base_year
valid_kb['Revenue_pred'] = base_rev * (growth_rev ** valid_kb['years_ahead']) * valid_kb['seasonal_rev_norm']

kaggle_mae = float(mean_absolute_error(valid_kb['Revenue'], valid_kb['Revenue_pred']))
print('Kaggle-baseline holdout MAE:', kaggle_mae)

plt.plot(valid_kb['Date'], valid_kb['Revenue'], label='Actual', linewidth=1)
plt.plot(valid_kb['Date'], valid_kb['Revenue_pred'], '--', label='Kaggle baseline pred', linewidth=1)
plt.title('Holdout: Actual vs Kaggle baseline')
plt.legend()
plt.show()

## B) Train theo code đã tích hợp trực tiếp trong notebook (Time-Series CV)

In [ ]:
def _date_features(dates: pd.Series, origin: pd.Timestamp) -> pd.DataFrame:
    days_since = (dates - origin).dt.days.astype(float)
    day_of_week = dates.dt.dayofweek.astype(float)
    month = dates.dt.month.astype(float)
    day_of_year = dates.dt.dayofyear.astype(float)
    week_sin = np.sin(2.0 * np.pi * day_of_week / 7.0)
    week_cos = np.cos(2.0 * np.pi * day_of_week / 7.0)
    year_sin = np.sin(2.0 * np.pi * day_of_year / 365.25)
    year_cos = np.cos(2.0 * np.pi * day_of_year / 365.25)
    return pd.DataFrame(
        {
            'days_since': days_since,
            'day_of_week': day_of_week,
            'month': month,
            'week_sin': week_sin,
            'week_cos': week_cos,
            'year_sin': year_sin,
            'year_cos': year_cos,
        }
    )

def _fit_linear_regression(train_df: pd.DataFrame, predict_dates: pd.Series) -> np.ndarray:
    origin = train_df['Date'].min()
    x_train = _date_features(train_df['Date'], origin)
    y_train = train_df['Revenue'].to_numpy()
    model = LinearRegression()
    model.fit(x_train, y_train)
    x_predict = _date_features(predict_dates, origin)
    return model.predict(x_predict)

def _seasonal_naive_predict(history_df: pd.DataFrame, predict_dates: pd.Series) -> np.ndarray:
    known = {row.Date: float(row.Revenue) for row in history_df.itertuples(index=False)}
    fallback = float(history_df['Revenue'].iloc[-1])
    predictions = []
    for date in pd.to_datetime(predict_dates).sort_values():
        lag_date = date - pd.Timedelta(days=7)
        if lag_date in known:
            pred = known[lag_date]
        else:
            past_dates = [d for d in known if d < date]
            pred = known[max(past_dates)] if past_dates else fallback
        predictions.append(float(pred))
        known[date] = float(pred)
    ordered = pd.DataFrame({'Date': pd.to_datetime(predict_dates), 'pred': predictions})
    return ordered.sort_values('Date')['pred'].to_numpy()

def _evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float('nan')
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

def _predict_with_method(train_df: pd.DataFrame, predict_dates: pd.Series, method: str) -> np.ndarray:
    if method == 'linear_regression':
        return _fit_linear_regression(train_df, predict_dates)
    if method == 'seasonal_naive':
        return _seasonal_naive_predict(train_df, predict_dates)
    raise ValueError(f'Unsupported method: {method}')

def _generate_expanding_window_splits(sales_df: pd.DataFrame, n_splits: int, valid_size: int, min_train_size: int):
    if n_splits < 1 or valid_size < 1 or min_train_size < 1:
        raise ValueError('cv_folds/cv_valid_size/cv_min_train_size phải >= 1')
    required_rows = min_train_size + n_splits * valid_size
    if len(sales_df) < required_rows:
        raise ValueError(f'Không đủ dữ liệu cho CV: cần >= {required_rows}, hiện có {len(sales_df)}')
    splits = []
    for fold_idx in range(n_splits):
        train_end = min_train_size + fold_idx * valid_size
        valid_start = train_end
        valid_end = valid_start + valid_size
        splits.append((train_end, valid_start, valid_end))
    return splits

def _cross_validate_methods(sales_df: pd.DataFrame, methods: list[str], n_splits: int, valid_size: int, min_train_size: int):
    fold_windows = _generate_expanding_window_splits(sales_df, n_splits, valid_size, min_train_size)
    cv_results = {}
    for method in methods:
        fold_metrics = []
        for fold_number, (train_end, valid_start, valid_end) in enumerate(fold_windows, start=1):
            train_df = sales_df.iloc[:train_end].copy()
            valid_df = sales_df.iloc[valid_start:valid_end].copy()
            y_valid = valid_df['Revenue'].to_numpy()
            valid_pred = _predict_with_method(train_df, valid_df['Date'], method)
            metrics = _evaluate(y_valid, valid_pred)
            fold_metrics.append(
                {
                    'fold': fold_number,
                    'train_rows': int(train_df.shape[0]),
                    'valid_rows': int(valid_df.shape[0]),
                    **metrics,
                }
            )
        avg_metrics = {
            'mae': float(np.mean([float(f['mae']) for f in fold_metrics])),
            'rmse': float(np.mean([float(f['rmse']) for f in fold_metrics])),
            'r2': float(np.mean([float(f['r2']) for f in fold_metrics])),
        }
        cv_results[method] = {
            'n_folds': n_splits,
            'valid_size': valid_size,
            'min_train_size': min_train_size,
            'avg_metrics': avg_metrics,
            'fold_metrics': fold_metrics,
        }
    return cv_results

def train_time_series_model_inline(
    sales_df: pd.DataFrame,
    template_df: pd.DataFrame,
    output_dir: Path,
    method: str = 'auto',
    cv_folds: int = 5,
    cv_valid_size: int = 30,
    cv_min_train_size: int = 180,
):
    methods = ['linear_regression', 'seasonal_naive']
    cv_results = _cross_validate_methods(
        sales_df,
        methods=methods,
        n_splits=cv_folds,
        valid_size=cv_valid_size,
        min_train_size=cv_min_train_size,
    )
    linear_metrics = cv_results['linear_regression']['avg_metrics']
    naive_metrics = cv_results['seasonal_naive']['avg_metrics']

    if method == 'auto':
        selected_method = 'linear_regression' if float(linear_metrics['mae']) <= float(naive_metrics['mae']) else 'seasonal_naive'
    else:
        selected_method = method
    if selected_method not in methods:
        raise ValueError(f'Unsupported method: {selected_method}')

    revenue_pred = _predict_with_method(sales_df, template_df['Date'], selected_method)
    submission = template_df.copy()
    submission['Revenue'] = np.maximum(revenue_pred, 0.0)
    if 'COGS' not in submission.columns:
        if 'COGS' in sales_df.columns:
            ratio = float((sales_df['COGS'] / sales_df['Revenue']).replace([np.inf, -np.inf], np.nan).dropna().mean())
            if np.isnan(ratio):
                ratio = 0.1
        else:
            ratio = 0.1
        submission['COGS'] = submission['Revenue'] * ratio

    output_dir.mkdir(parents=True, exist_ok=True)
    submission_file = output_dir / 'submission.csv'
    metrics_file = output_dir / 'baseline_metrics.json'
    submission.to_csv(submission_file, index=False)

    payload = {
        'selected_method': selected_method,
        'selected_metrics': cv_results[selected_method]['avg_metrics'],
        'linear_regression_metrics': linear_metrics,
        'seasonal_naive_metrics': naive_metrics,
        'cv': {
            'strategy': 'expanding_window',
            'config': {
                'folds': cv_folds,
                'valid_size': cv_valid_size,
                'min_train_size': cv_min_train_size,
            },
            'results': cv_results,
        },
        'submission_path': str(submission_file),
    }
    metrics_file.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return payload

print('Đã tích hợp xong code train Time-Series CV trực tiếp trong notebook.')

In [ ]:
result = train_time_series_model_inline(
    sales_df=sales,
    template_df=template,
    output_dir=PROCESSED_DIR,
    method='auto',
    cv_folds=5,
    cv_valid_size=30,
    cv_min_train_size=180,
)

print(json.dumps(result, indent=2)[:1200], '...')

In [ ]:
forecasting_mae = float(result['selected_metrics']['mae'])
selected_method = result['selected_method']

compare_df = pd.DataFrame([
    {'approach': 'kaggle_baseline_holdout', 'mae': kaggle_mae},
    {'approach': f'forecasting_cv_selected({selected_method})', 'mae': forecasting_mae},
]).sort_values('mae')

display(compare_df)
print('Gợi ý: MAE thấp hơn thường đáng ưu tiên.')

In [ ]:
# Tạo thêm 1 file submission riêng theo Kaggle-baseline để tiện đối chiếu
test_kb = template.copy()
test_kb['month'] = test_kb['Date'].dt.month
test_kb['day'] = test_kb['Date'].dt.day
test_kb['year'] = test_kb['Date'].dt.year
test_kb['years_ahead'] = test_kb['year'] - base_year
test_kb = test_kb.merge(seasonal, on=['month', 'day'], how='left')
test_kb['rev_norm'] = test_kb['rev_norm'].fillna(1.0)
test_kb['cogs_norm'] = test_kb['cogs_norm'].fillna(1.0)

test_kb['Revenue'] = np.maximum(base_rev * (growth_rev ** test_kb['years_ahead']) * test_kb['rev_norm'], 0.0)
test_kb['COGS'] = np.maximum(base_cogs * (growth_cogs ** test_kb['years_ahead']) * test_kb['cogs_norm'], 0.0)

kaggle_sub = test_kb[['Date', 'Revenue', 'COGS']].copy()
kaggle_out = PROCESSED_DIR / 'submission_kaggle_baseline.csv'
kaggle_sub.to_csv(kaggle_out, index=False)

print('Saved:', kaggle_out)
display(kaggle_sub.head(5))

## Kết quả sau khi chạy notebook

- Code train CV đã được tích hợp trực tiếp trong notebook này và sẽ ghi:
  - `data/processed/submission.csv`
  - `data/processed/baseline_metrics.json`
- Notebook này ghi thêm:
  - `data/processed/submission_kaggle_baseline.csv`

Bạn có thể upload từng file submission để so sánh leaderboard.